In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from PIL import Image
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP & TRANSFORMS
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🦖 Starting DINOv2 Fine-Tuning on {DEVICE}...")

# DINOv2 specific transforms
transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset class to handle the HF list format
class BrainTumorDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform
        # Create label mapping if strings
        unique_labels = sorted(list(set(hf_dataset['label'])))
        if isinstance(unique_labels[0], str):
            self.label_map = {l: i for i, l in enumerate(unique_labels)}
        else:
            self.label_map = None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img_data = sample['image']
        label = sample['label']
        
        if isinstance(img_data, list):
            img = Image.fromarray(np.array(img_data, dtype=np.uint8).reshape(224, 224, 3))
        else:
            img = img_data
            
        if self.label_map:
            label = self.label_map[label]
            
        return self.transform(img), label

# Load Datasets
print("📥 Loading BRISC dataset...")
train_ds = BrainTumorDataset(load_dataset('srijan2715/brisc2025-paper-pipeline', split='train'), transform)
val_ds = BrainTumorDataset(load_dataset('srijan2715/brisc2025-paper-pipeline', split='val'), transform)
test_ds = BrainTumorDataset(load_dataset('srijan2715/brisc2025-paper-pipeline', split='test'), transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 2. LOAD & CONFIGURE DINOv2 FOR FINE-TUNING
# ==========================================
print("🧠 Loading DINOv2 ViT-B/14...")
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(DEVICE)

# Add classification head
model.head = nn.Linear(768, 4).to(DEVICE)

# Freeze all layers EXCEPT the last 2 transformer blocks and the head
for name, param in model.named_parameters():
    param.requires_grad = False

# Unfreeze the last 2 blocks (blocks 10 and 11 out of 12) and the head
for name, param in model.named_parameters():
    if 'blocks.10.' in name or 'blocks.11.' in name or 'head.' in name or 'norm.' in name:
        param.requires_grad = True

print(f"✅ Model configured. Fine-tuning {sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters.")

# ==========================================
# 3. TRAINING LOOP
# ==========================================
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

best_val_acc = 0.0
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    scheduler.step()
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, preds = torch.max(model(x), 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(y.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'dinov2_finetuned_best.pth')

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")

# ==========================================
# 4. FINAL TEST EVALUATION
# ==========================================
print("📊 Evaluating on Test Set...")
model.load_state_dict(torch.load('dinov2_finetuned_best.pth'))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, preds = torch.max(model(x), 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y.cpu().numpy())

acc = accuracy_score(test_labels, test_preds)
prec = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
rec = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

print("\n" + "="*50)
print("🏆 DINOv2 FINE-TUNED RESULTS (BRISC)")
print("="*50)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)
print("Task Complete")

🦖 Starting DINOv2 Fine-Tuning on cuda...
📥 Loading BRISC dataset...
🧠 Loading DINOv2 ViT-B/14...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


✅ Model configured. Fine-tuning 14183428 parameters.
Epoch 1/10 | Loss: 0.3235 | Val Acc: 0.9613
Epoch 2/10 | Loss: 0.0623 | Val Acc: 0.9747
Epoch 3/10 | Loss: 0.0202 | Val Acc: 0.9840
Epoch 4/10 | Loss: 0.0063 | Val Acc: 0.9840
Epoch 5/10 | Loss: 0.0053 | Val Acc: 0.9813
Epoch 6/10 | Loss: 0.0024 | Val Acc: 0.9907
Epoch 7/10 | Loss: 0.0008 | Val Acc: 0.9907
Epoch 8/10 | Loss: 0.0006 | Val Acc: 0.9893
Epoch 9/10 | Loss: 0.0006 | Val Acc: 0.9893
Epoch 10/10 | Loss: 0.0005 | Val Acc: 0.9907

🏆 Best Validation Accuracy: 0.9907
📊 Evaluating on Test Set...

🏆 DINOv2 FINE-TUNED RESULTS (BRISC)
Accuracy:  0.9810
Precision: 0.9811
Recall:    0.9810
F1-Score:  0.9810
✅ Task 7 COMPLETE! All demands for Part B are now finished.
